# Demo 1: Grover on 3 qubits
8 possible passwords, one is correct. How many guesses does a quantum computer need?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from demo_utils import plot_probs, plot_counts, load_grover, GREEN, LIGHT

n = 3
dev = qml.device("default.qubit")

### The secret: a number from 0 to 7

In [ ]:
secret = 5   # the audience's number
secret_bits = [int(b) for b in format(secret, f"0{n}b")]
print("secret:", secret, "in binary:", "".join(map(str, secret_bits)))

### The algorithm: superposition, then repeat oracle + diffusion

In [ ]:
@qml.qnode(dev)
def grover(iterations):
    for w in range(n):
        qml.Hadamard(wires=w)                        # try all 8 at once
    for _ in range(iterations):
        qml.FlipSign(secret_bits, wires=range(n))    # oracle: flip the sign of the secret
        qml.GroverOperator(wires=range(n))           # diffusion: reflect about the mean
    return qml.probs(wires=range(n))

target = format(secret, f"0{n}b")
plot_probs(grover(0), title="0 iterations: every answer equally likely", highlight=target)
plot_probs(grover(2), title="2 iterations: the secret stands out", highlight=target)

### More is not better
The optimum is about π/4 · √8 ≈ 2 iterations. After that the state rotates *past* the answer.
(Keep going and it eventually comes back around, but that costs several times more steps, and on real hardware every extra step adds noise.)

In [ ]:
ks = np.arange(0, 5)
p_sim = [grover(int(k))[secret] for k in ks]

theta = np.arcsin(1 / np.sqrt(2 ** n))
kk = np.linspace(0, 4, 200)
p_theory = np.sin((2 * kk + 1) * theta) ** 2

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(kk, p_theory, color=LIGHT, lw=2, label="theory: sin²((2k+1)θ)")
ax.scatter(ks, p_sim, color=GREEN, s=50, zorder=3, label="simulation")
ax.set_xlabel("Grover iterations k")
ax.set_ylabel("P(secret)")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, loc="upper right")
ax.set_title("Success probability rotates, it doesn't just grow")
plt.show()

### Why is the speedup quadratic?
- Grover lives in a 2-D plane: **w** = the secret, **r** = all wrong answers.
- The start **s** is at angle θ from **r**, with sin θ = 1/√N (N = 8: θ ≈ 20.7°).
- Oracle + diffusion = two reflections = a **rotation by 2θ**. So P = sin²((2k+1)θ): the curve above.
- Reaching **w** takes about 90° / 2θ ≈ (π/4)·√N steps: **√N instead of N**.
- √N is optimal for unstructured search (Bennett, Bernstein, Brassard, Vazirani 1997).

**But:** each of the 9.3 million steps for the slide 9 password runs the whole password check as an error-corrected quantum circuit, one after another, and Grover barely parallelises (M machines give only √M). A GPU checks billions of passwords per second. For this password, GPUs win, today and for the foreseeable future.

In [ ]:
# The geometry for N = 8: start at θ, each iteration adds 2θ
N = 2 ** n
fig, ax = plt.subplots(figsize=(6, 5.4))
ax.set_aspect("equal"); ax.axis("off")
ax.annotate("", xy=(1.15, 0), xytext=(-0.45, 0), arrowprops=dict(arrowstyle="-|>", color="grey", lw=1.5))
ax.annotate("", xy=(0, 1.2), xytext=(0, -0.05), arrowprops=dict(arrowstyle="-|>", color="grey", lw=1.5))
ax.text(1.15, -0.08, "r: wrong answers", ha="right", va="top", color="grey")
ax.text(0.03, 1.2, "w: the secret", va="center", color=GREEN)

shades = [LIGHT, "#44B749", GREEN, "#D6D6D6"]   # 2 iterations = the best state
for k in range(4):
    a = (2 * k + 1) * theta
    p = np.sin(a) ** 2
    ax.annotate("", xy=(np.cos(a), np.sin(a)), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=shades[k], lw=3 if k < 3 else 2))
    label = "start s" if k == 0 else f"{k} iteration{'s' if k > 1 else ''}"
    ha = "left" if np.cos(a) > 0.1 else "right"
    ax.text(1.08 * np.cos(a), 1.08 * np.sin(a), f"{label}: P = {p:.3g}", ha=ha, va="center", fontsize=10)

t = np.linspace(0, theta, 30)
ax.plot(0.25 * np.cos(t), 0.25 * np.sin(t), color="black", lw=1)
ax.text(0.3 * np.cos(theta / 2), 0.3 * np.sin(theta / 2), "θ", va="center")
t = np.linspace(theta, 5 * theta, 60)
ax.plot(0.6 * np.cos(t), 0.6 * np.sin(t), color=GREEN, ls="--", lw=1.5)
ax.set_xlim(-1.35, 1.45); ax.set_ylim(-0.15, 1.3)
ax.set_title(f"N = {N}: θ = arcsin(1/√N) ≈ {np.degrees(theta):.1f}°, each iteration rotates by 2θ")
plt.show()

In [ ]:
# The same geometry at scale: steps grow like √N, not N
# (for 26^10: 9.3 million sequential, error-corrected steps)
print(f"{'N':>22} {'classical (worst case)':>24} {'Grover ≈ (π/4)·√N':>20}")
for label, N in [("8 (this demo)", 8), ("1 million", 10**6),
                 ("26^10 (slide 9)", 26**10), ("2^64", 2**64)]:
    print(f"{label:>22} {float(N):>24.3g} {np.pi / 4 * np.sqrt(float(N)):>20.3g}")

### Sampling like a real QPU: 1,000 shots

In [ ]:
@qml.set_shots(1000)
@qml.qnode(dev)
def grover_shots(iterations):
    for w in range(n):
        qml.Hadamard(wires=w)
    for _ in range(iterations):
        qml.FlipSign(secret_bits, wires=range(n))
        qml.GroverOperator(wires=range(n))
    return qml.counts(wires=range(n))

plot_counts(grover_shots(2), title="2 iterations, 1,000 shots", highlight=target)

### The same circuit on a real IBM quantum processor (1,000 shots)

In [ ]:
counts, backend, date, bits = load_grover(secret)
plot_counts(counts, title=f"Grover, secret {bits}, on {backend} ({date})", highlight=bits)